In [ ]:
from jax import config as jax_config
jax_config.update("jax_enable_x64", True)

import matplotlib.pyplot as plt

import colors
import helpers_builders
import plot_funcs
from config import CFG
from EquilibriumClass import EquilibriumClass
from StateClass import StateClass
from SupervisorClass import SupervisorClass
from VariablesClass import VariablesClass

In [ ]:
colors_lst, red, custom_cmap = colors.color_scheme()
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=colors_lst)

## Configured model

All user-editable parameters are in `config.py`.

In [ ]:
Variabs = VariablesClass(CFG, plot_potential=CFG.Output.plot_potential)
Sprvsr = SupervisorClass(CFG)
State = StateClass(Variabs)
State.get_free_dof_parameters(Variabs)
Eq = EquilibriumClass(Variabs)

## Input pulse

In [ ]:
impulse_data = Sprvsr.program_impulse()
plot_funcs.plot_impulse(Sprvsr.timepoints, impulse_data)

## Simulations

In [ ]:
# empty tuple for simulating same pulse, different initial phase
solutions_by_phase = {}

# loop initial phase
for initial_phase in Sprvsr.initial_phases:
    State.get_current_local_global_states(Variabs, initial_phase)
    sol_free = Eq.solve(State, Sprvsr)
    sol_global = State.reshape_local_to_global(Variabs, Sprvsr, sol_free)
    solutions_by_phase[initial_phase] = sol_global
    if CFG.Output.plot_responses:
        plot_funcs.plot_response(
            sol_global, Sprvsr.timepoints, sup_title=f"Initial phase {initial_phase}",
            spring_stiffness=Variabs.k1,
        )

In [ ]:
State.state0_global

## State and endpoint-force comparison

In [ ]:
for initial_phase, sol_global in solutions_by_phase.items():
    final_relative_displacement = (
        sol_global[-1, 2, 1:-1] - sol_global[-1, 0, 1:-1]
    )
    final_state = State.get_system_state(
        final_relative_displacement, threshold=CFG.Output.state_threshold
    )
    print(
        f"Initial {initial_phase} -> final state: {final_state} "
        f"({helpers_builders.state_to_number(final_state)})"
    )

if CFG.Output.compare_endpoint_forces and len(solutions_by_phase) == 2:
    _, _, force_delay = plot_funcs.plot_force_comparison(
        solutions_by_phase, Sprvsr.timepoints, Variabs.k1, Sprvsr.start_time,
        CFG.Output.force_arrival_threshold_fraction,
    )
    print(f"Force-arrival delay: {force_delay * 1e3:.1f} ms")